In [13]:
import torch

In [ ]:
inputs = torch.tensor(
  [[0.43, 0.15, 0.89], # Your     (x^1)
   [0.55, 0.87, 0.66], # journey  (x^2)
   [0.57, 0.85, 0.64], # starts   (x^3)
   [0.22, 0.58, 0.33], # with     (x^4)
   [0.77, 0.25, 0.10], # one      (x^5)
   [0.05, 0.80, 0.55]] # step     (x^6)
)

In [ ]:
import torch.nn as nn
class SelfAttention_v2(nn.Module):

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)

        context_vec = attn_weights @ values
        return context_vec

In [ ]:
din = 3
dout = 2

In [ ]:
context_length = inputs[0].size()[0]
one_mat = torch.ones(context_length, context_length)

In [ ]:
self_attended = SelfAttention_v2(din, dout)
queries = self_attended.W_query(inputs)
keys = self_attended.W_key(inputs)
values = self_attended.W_value(inputs)
attention_scores = queries @ keys.T
attention_weights = torch.softmax(attention_scores / keys.shape[0] ** 0.5, dim = -1)
print(attention_weights)

tensor([[0.1604, 0.1676, 0.1674, 0.1694, 0.1642, 0.1711],
        [0.1615, 0.1669, 0.1670, 0.1683, 0.1687, 0.1675],
        [0.1615, 0.1669, 0.1670, 0.1684, 0.1687, 0.1676],
        [0.1644, 0.1667, 0.1668, 0.1672, 0.1685, 0.1664],
        [0.1626, 0.1670, 0.1670, 0.1682, 0.1669, 0.1682],
        [0.1644, 0.1666, 0.1667, 0.1672, 0.1692, 0.1659]],
       grad_fn=<SoftmaxBackward0>)


In [ ]:
context_length = attention_scores.shape[0]
simple_mask = torch.tril(torch.ones(context_length, context_length))
print(simple_mask)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [ ]:
masked_attention_weights = attention_weights * simple_mask
masked_attention_weights = masked_attention_weights / masked_attention_weights.sum(dim = 1)
print(masked_attention_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [1.0067, 0.5083, 0.0000, 0.0000, 0.0000, 0.0000],
        [1.0065, 0.5083, 0.3371, 0.0000, 0.0000, 0.0000],
        [1.0251, 0.5075, 0.3366, 0.2515, 0.0000, 0.0000],
        [1.0135, 0.5086, 0.3371, 0.2528, 0.2007, 0.0000],
        [1.0248, 0.5072, 0.3365, 0.2513, 0.2034, 0.1659]],
       grad_fn=<DivBackward0>)


In [ ]:
# Now make it properly causal
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked_attention_scores  = attention_scores.masked_fill(mask.bool(), -torch.inf)
masked_attention_weights = torch.softmax(masked_attention_scores / keys.shape[-1] ** 0.5 , dim = 1)
dropout = nn.Dropout(0.5)
masked_attention_weights = dropout(masked_attention_weights)
print(masked_attention_weights)
context_vector = masked_attention_weights @ values
print(context_vector)

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.9713, 1.0287, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6411, 0.0000, 0.6797, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.5025, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.4076, 0.0000, 0.0000],
        [0.3255, 0.0000, 0.3335, 0.3351, 0.0000, 0.3308]],
       grad_fn=<MulBackward0>)
tensor([[ 0.0000,  0.0000],
        [-0.4367,  0.6254],
        [-0.2976,  0.4003],
        [-0.0307,  0.2011],
        [ 0.0296,  0.1163],
        [-0.0605,  0.4530]], grad_fn=<MmBackward0>)


In [ ]:
batch = torch.stack((inputs, inputs), dim = 0)
print(batch.shape)

torch.Size([2, 6, 3])


In [ ]:
# Causal Attention Class
class CausalAttention(nn.Module):
  def __init__(self, din, dout, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.Wq = nn.Linear(din, dout, bias=qkv_bias)
    self.Wk = nn.Linear(din, dout, bias=qkv_bias)
    self.Wv = nn.Linear(din, dout, bias=qkv_bias)
    self.dropout = nn.Dropout(dropout)
    self.mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)

  def forward(self, x):
    b, num_tokens, din = x.shape
    queries = self.Wq(x)
    keys = self.Wk(x)
    values = self.Wv(x)
    attention_scores = queries @ keys.transpose(1, 2)
    attention_scores = attention_scores.masked_fill(self.mask.bool(), -torch.inf)
    attention_weights = torch.softmax(attention_scores / keys.shape[-1] ** 0.5, dim = -1)
    attention_weights = self.dropout(attention_weights)
    context_vector = attention_weights @ values
    return context_vector



In [ ]:
cat = CausalAttention(din, dout, context_length, 0.05)
context_vector = cat.forward(batch)
print(context_vector)

tensor([[[-0.1019,  0.3293],
         [ 0.0291,  0.2316],
         [ 0.0799,  0.2028],
         [ 0.0768,  0.1500],
         [ 0.1121,  0.1558],
         [ 0.1073,  0.1462]],

        [[-0.1019,  0.3293],
         [ 0.0291,  0.2316],
         [ 0.0799,  0.2028],
         [ 0.0768,  0.1500],
         [ 0.1495,  0.1856],
         [ 0.0346,  0.0791]]], grad_fn=<UnsafeViewBackward0>)


In [16]:
class MultiHeadAttentionWrapper(nn.Module):
  def __init__(self, din, dout, num_heads, context_length, dropout, qkv_bias=False):
    super().__init__()
    self.heads = nn.ModuleList(
            [CausalAttention(din, dout, context_length, dropout, qkv_bias)
             for _ in range(num_heads)]
        )

  def forward(self, x):
    return torch.cat([head(x) for head in self.heads], dim = -1)

In [17]:
mha = MultiHeadAttentionWrapper(din, dout, 2, context_length, 0.0)
context = mha.forward(batch)
print(context)

tensor([[[-0.0821,  0.1589,  0.0063,  0.0288],
         [-0.0453,  0.3488, -0.1603,  0.0829],
         [-0.0332,  0.4054, -0.2070,  0.0960],
         [-0.0238,  0.3903, -0.2174,  0.1001],
         [-0.0189,  0.3393, -0.1587,  0.0598],
         [-0.0155,  0.3673, -0.2057,  0.0892]],

        [[-0.0821,  0.1589,  0.0063,  0.0288],
         [-0.0453,  0.3488, -0.1603,  0.0829],
         [-0.0332,  0.4054, -0.2070,  0.0960],
         [-0.0238,  0.3903, -0.2174,  0.1001],
         [-0.0189,  0.3393, -0.1587,  0.0598],
         [-0.0155,  0.3673, -0.2057,  0.0892]]], grad_fn=<CatBackward0>)
